# 数据质量检测

先确认 9 张表的规模、缺失和重复情况，再决定哪些字段能直接用。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

## 各表规模

In [2]:
sql = """
SELECT 'orders' AS tbl, COUNT(*) AS n FROM olist_orders_dataset
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_dataset
UNION ALL SELECT 'order_payments', COUNT(*) FROM olist_order_payments_dataset
UNION ALL SELECT 'order_reviews', COUNT(*) FROM olist_order_reviews_dataset
UNION ALL SELECT 'customers', COUNT(*) FROM olist_customers_dataset
UNION ALL SELECT 'products', COUNT(*) FROM olist_products_dataset
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_dataset
UNION ALL SELECT 'geolocation', COUNT(*) FROM olist_geolocation_dataset
UNION ALL SELECT 'category_translation', COUNT(*) FROM product_category_name_translation
ORDER BY n DESC
"""
query(sql)

,tbl,n
0,geolocation,1000163
1,order_items,112650
2,order_payments,103886
3,orders,99441
4,customers,99441
5,order_reviews,99224
6,products,32951
7,sellers,3095
8,category_translation,71


## orders 的缺失情况

未送达的订单本来就没有送达时间，不算脏数据。

In [3]:
query("""
SELECT COUNT(*) AS total,
       SUM(order_approved_at IS NULL)            AS approved_null,
       SUM(order_delivered_carrier_date IS NULL) AS carrier_null,
       SUM(order_delivered_customer_date IS NULL) AS delivered_null,
       SUM(order_estimated_delivery_date IS NULL) AS estimated_null
FROM olist_orders_dataset
""")

,total,approved_null,carrier_null,delivered_null,estimated_null
0,99441,160,1783,2965,0


## 订单状态分布

In [4]:
query("""
SELECT order_status,
       COUNT(*) AS orders,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS pct
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY orders DESC
""")

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


## 重复与异常

geolocation 里同一个邮编有大量重复坐标；order_items 里有零价和运费为负的记录。

In [5]:
query("""
SELECT
  (SELECT COUNT(*) FROM olist_geolocation_dataset) AS geo_rows,
  (SELECT COUNT(DISTINCT geolocation_zip_code_prefix) FROM olist_geolocation_dataset) AS geo_zip,
  (SELECT COUNT(*) FROM olist_order_reviews_dataset) AS review_rows,
  (SELECT COUNT(DISTINCT order_id) FROM olist_order_reviews_dataset) AS review_orders
""")

,geo_rows,geo_zip,review_rows,review_orders
0,1000163,19015,99224,98673


In [6]:
query("""
SELECT COUNT(*) AS items,
       SUM(price <= 0) AS price_le_0,
       MIN(price) AS min_price,
       MAX(price) AS max_price,
       SUM(freight_value < 0) AS freight_negative
FROM olist_order_items_dataset
""")

,items,price_le_0,min_price,max_price,freight_negative
0,112650,0,0.85,6735.0,0
